# Day 16: Building Custom Tools

Walkthrough of what makes a tool good, turning functions into tools automatically, domain-specific tools, common mistakes, testing, wiring tools into the Day 15 agent, and safety.

## 1. What makes a good tool

A tool is a function with a name, a description, and clear inputs. The agent only ever sees the label, never the implementation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-15-agentic-ai'))
print("One job per tool. Clear name. Clear parameters. Helpful errors.")
print("A vague or incomplete tool description produces a confused agent -- not a bad agent.")

## 2. Turn a function into a tool

Using `inspect` to read a function's real name, docstring, and parameters, so the agent-facing documentation can never drift from the actual signature.

In [ ]:
from tool_registry import ToolRegistry
from tool_builder import build_tool_from_function, add_numbers, greet_customer

registry = ToolRegistry()
for fn in (add_numbers, greet_customer):
    build_tool_from_function(registry, fn)

print(registry.get_documentation())
print()
print(registry.call("add_numbers", 2, 3))

## 3. Support tools

Order lookup, refund eligibility, and ticket creation against a small simulated dataset -- each validating its inputs and returning a structured result.

In [ ]:
from support_tools import lookup_order, check_refund_eligibility, create_ticket

print(lookup_order("ORD-1001"))
print(check_refund_eligibility("ORD-1003"))
print(create_ticket("Item arrived damaged", priority="high"))

## 4. Shopping tools

Product search, price comparison, and cart management against a small simulated catalog.

In [ ]:
from shopping_tools import search_products, compare_prices, add_to_cart

print(search_products("usb"))
print(compare_prices("SKU-1", "SKU-2"))
print(add_to_cart("SKU-3", 2))
print("Out of stock, fails on purpose:", add_to_cart("SKU-2", 1))

## 5. Bad vs good tool design

Four common mistakes -- a tool that does too much, a vague name, unclear parameters, a messy result -- each with a before/after fix.

In [ ]:
from tool_design_mistakes import handle_order_bad, lookup_order_status, refund_order
from tool_design_mistakes import process, convert_to_uppercase
from tool_design_mistakes import get_customer_info_bad, get_customer_info

print("Bad (does too much):", handle_order_bad("ORD-1", "refund", refund_amount=20))
print("Fixed (one job):     ", refund_order("ORD-1", 20))
print()
print("Bad (vague name): process('hello') ->", process("hello"))
print("Fixed (clear name): convert_to_uppercase('hello') ->", convert_to_uppercase("hello"))
print()
print("Bad (messy result):", get_customer_info_bad("CUST-1"))
print("Fixed (structured): ", get_customer_info("CUST-1"))

## 6. Testing tools

Testing valid input, missing input, wrong types, and edge cases before an agent ever calls the tool unattended.

In [ ]:
from tool_testing import run_test
from support_tools import lookup_order
from shopping_tools import add_to_cart

results = [
    run_test("lookup_order with a real ID", lookup_order, ["ORD-1001"], expect_ok=True),
    run_test("lookup_order with an empty string", lookup_order, [""], expect_ok=False),
    run_test("add_to_cart with quantity zero", add_to_cart, ["SKU-1", 0], expect_ok=False),
]
print(f"\n{sum(results)}/{len(results)} tests passed")

## 7. Agent using custom tools

The Day 15 agent loop (think, decide, act, observe) driven by today's support tools instead of Day 15's originals.

In [ ]:
from agent_with_custom_tools import run_agent
from tool_builder import build_tool_from_function
from support_tools import lookup_order, check_refund_eligibility, create_ticket

agent_registry = ToolRegistry()
for fn in (lookup_order, check_refund_eligibility, create_ticket):
    build_tool_from_function(agent_registry, fn)

history = run_agent(agent_registry, "Check order ORD-1001 and open a ticket if it's damaged.")
print(f"\nTotal tool calls: {len(history)}")

## 8. Safety

Validating inputs before acting, refusing to run arbitrary code, requiring confirmation for risky actions, and logging every call.

In [ ]:
from tool_safety import validate_before_acting, process_refund, CALL_LOG

print(validate_before_acting(50, max_allowed=100))
print(process_refund("ORD-1001", 40.00))
print(process_refund("ORD-1002", 500.00))
print(process_refund("ORD-1002", 500.00, confirmed=True))
print(f"\nCall log entries: {len(CALL_LOG)}")

## 9. Key takeaways

- A tool is a function plus a name, description, and parameter list -- the agent only ever sees the label, not the code.
- Generating that label from the real function signature (via `inspect`) keeps it from ever drifting out of sync.
- One job per tool, a clear name, clear parameters, and a structured result make a tool easy for an agent to use correctly.
- A tool that does too much, is vaguely named, has unclear parameters, or returns an unstructured blob will genuinely confuse an agent, not just annoy a human reader.
- Tools should be tested against valid input, missing input, wrong types, and edge cases BEFORE an agent ever calls them unattended.
- Safety is load-bearing: validate before acting, never execute arbitrary code, require confirmation for risky actions, and log every call.
- None of this changes Day 15's agent loop -- it only changes the quality of what the agent has available to act with.